# ChemBreak Task Bank Generator V5
A simplified four-family pipeline for generating HarmBench-style base chemistry tasks while preserving the V4 output schemas.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
BRANCH = "main"
PROJECT_SUBDIR = "ChemBreak_TaskBank_Generator_v5"
CLONE_DIR = Path("/content/ChemBreak")

if not CLONE_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(CLONE_DIR)], check=True)
PROJECT_DIR = CLONE_DIR / PROJECT_SUBDIR
print(PROJECT_DIR)

In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements_colab.txt")], check=True)

## Optional Hugging Face login
Gemma requires accepted access conditions and an authenticated Hugging Face account.

In [ ]:
USE_HF_LOGIN = False
if USE_HF_LOGIN:
    from huggingface_hub import notebook_login
    notebook_login()

## Offline structural smoke test
This uses no model download and writes to `outputs_smoke`.

In [ ]:
subprocess.run([
    "python", str(PROJECT_DIR / "chembreak_v5.py"),
    "--project-dir", str(PROJECT_DIR),
    "--config", "config_smoke.json",
    "--stage", "all",
    "--backend", "mock"
], check=True)

## Run the real four-family pilot
Each checkpoint loads sequentially. Outputs are checkpointed after every candidate and judgment.

In [ ]:
RUN_REAL_PILOT = False
if RUN_REAL_PILOT:
    subprocess.run([
        "python", str(PROJECT_DIR / "chembreak_v5.py"),
        "--project-dir", str(PROJECT_DIR),
        "--config", "config.json",
        "--stage", "all",
        "--backend", "hf"
    ], check=True)

In [ ]:
import pandas as pd
OUTPUT_DIR = PROJECT_DIR / "outputs"
if (OUTPUT_DIR / "provisional_task_bank.csv").exists():
    display(pd.read_csv(OUTPUT_DIR / "provisional_task_bank.csv"))
    display(pd.read_csv(OUTPUT_DIR / "generator_summary.csv"))

## Download the results
Run this after generation to download a zip archive from Colab.

In [ ]:
from google.colab import files
archive = subprocess.run(["zip", "-r", "/content/chembreak_v5_outputs.zip", str(OUTPUT_DIR)], check=True)
files.download("/content/chembreak_v5_outputs.zip")